# Retrieval Relevance Analysis

This notebook examines the relationship between user queries and the source chunks retrieved by the RAG system. It combines:

- **Quantitative analysis**: TF-IDF and semantic (embedding) similarity scores between each query and its retrieved chunks
- **Qualitative analysis**: Side-by-side display of queries and their source text for manual inspection

Use this to identify weak retrievals, over-relied-upon documents, and queries that the system struggles with.

In [ ]:
import json
import re
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display, HTML, Markdown

LOG_PATH = Path("../logs/chat_interactions.jsonl")

records = []
with open(LOG_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

# Filter to records whose sources include text
records_with_text = [
    r for r in records
    if any(s.get("text") for s in r.get("sources", []))
]

print(f"Total interactions: {len(records)}")
print(f"Interactions with source text: {len(records_with_text)}")
print(f"Interactions without source text: {len(records) - len(records_with_text)}")

## 1. Build Query-Source Pairs

Flatten each interaction into individual (query, source_text) pairs for analysis.

In [ ]:
pairs = []
for rec in records_with_text:
    question = rec["question"]
    intent = rec.get("intent", "unknown")
    session = rec["session_id"]
    ts = rec["timestamp"]

    # De-duplicate sources by parent_id within this interaction
    seen = set()
    for src in rec["sources"]:
        text = src.get("text", "").strip()
        pid = src.get("parent_id", "")
        if not text or pid in seen:
            continue
        seen.add(pid)
        pairs.append({
            "timestamp": ts,
            "session_id": session,
            "question": question,
            "intent": intent,
            "parent_id": pid,
            "title": src.get("title", ""),
            "collection": src.get("collection", ""),
            "source_text": text,
        })

pairs_df = pd.DataFrame(pairs)
print(f"Total unique query-source pairs: {len(pairs_df)}")
print(f"Unique questions: {pairs_df['question'].nunique()}")
print(f"Unique source documents: {pairs_df['parent_id'].nunique()}")

## 2. TF-IDF Keyword Overlap

Measures lexical similarity — how much vocabulary the query and retrieved chunk share. This catches cases where the retriever found topically unrelated text.

In [ ]:
def compute_tfidf_similarity(questions, source_texts):
    """Compute pairwise TF-IDF cosine similarity between each query and its source."""
    vectorizer = TfidfVectorizer(
        stop_words="english",
        max_features=10000,
        ngram_range=(1, 2),
    )
    # Fit on all texts together
    all_texts = list(questions) + list(source_texts)
    tfidf_matrix = vectorizer.fit_transform(all_texts)

    n = len(questions)
    q_vectors = tfidf_matrix[:n]
    s_vectors = tfidf_matrix[n:]

    # Pairwise similarity for each (query_i, source_i) pair
    sims = np.array([
        cosine_similarity(q_vectors[i:i+1], s_vectors[i:i+1])[0, 0]
        for i in range(n)
    ])
    return sims

pairs_df["tfidf_similarity"] = compute_tfidf_similarity(
    pairs_df["question"].values,
    pairs_df["source_text"].values,
)

print("TF-IDF Similarity Distribution:")
print(pairs_df["tfidf_similarity"].describe().round(3))

## 3. Semantic Similarity (Embedding-Based)

Uses the same embedding model as the RAG system (`BAAI/bge-small-en-v1.5`) to measure meaning-level similarity. This catches cases where vocabulary differs but the topic is the same — and vice versa.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-small-en-v1.5")

print("Encoding queries...")
q_embeddings = model.encode(pairs_df["question"].tolist(), show_progress_bar=True, batch_size=32)

print("Encoding source texts...")
s_embeddings = model.encode(pairs_df["source_text"].tolist(), show_progress_bar=True, batch_size=32)

# Pairwise cosine similarity
pairs_df["semantic_similarity"] = np.array([
    cosine_similarity(q_embeddings[i:i+1], s_embeddings[i:i+1])[0, 0]
    for i in range(len(pairs_df))
])

print("\nSemantic Similarity Distribution:")
print(pairs_df["semantic_similarity"].describe().round(3))

## 4. Combined Relevance Score

Average of TF-IDF and semantic similarity, giving a balanced view of both lexical and meaning-level relevance.

In [ ]:
pairs_df["combined_score"] = (pairs_df["tfidf_similarity"] + pairs_df["semantic_similarity"]) / 2

print("Combined Relevance Score Distribution:")
print(pairs_df["combined_score"].describe().round(3))

## 5. Per-Query Relevance Summary

Aggregate scores across all sources for each query. Queries with low average scores may indicate weak retrieval.

In [ ]:
query_scores = pairs_df.groupby(["question", "intent"]).agg(
    num_sources=("parent_id", "count"),
    avg_tfidf=("tfidf_similarity", "mean"),
    avg_semantic=("semantic_similarity", "mean"),
    min_semantic=("semantic_similarity", "min"),
    max_semantic=("semantic_similarity", "max"),
    avg_combined=("combined_score", "mean"),
).round(3).sort_values("avg_combined", ascending=True).reset_index()

print(f"Queries ranked by relevance (lowest first):")
query_scores.head(15).style.background_gradient(subset=["avg_combined"], cmap="RdYlGn")

In [ ]:
print("Queries with strongest retrieval:")
query_scores.tail(10).style.background_gradient(subset=["avg_combined"], cmap="RdYlGn")

## 6. Relevance by Intent

Are certain intent categories consistently better or worse at finding relevant sources?

In [ ]:
intent_relevance = pairs_df.groupby("intent").agg(
    num_pairs=("question", "count"),
    avg_tfidf=("tfidf_similarity", "mean"),
    avg_semantic=("semantic_similarity", "mean"),
    avg_combined=("combined_score", "mean"),
    std_combined=("combined_score", "std"),
).round(3)

intent_relevance

## 7. Relevance by Collection

Which document collections tend to produce more relevant chunks?

In [ ]:
collection_relevance = pairs_df.groupby("collection").agg(
    num_pairs=("question", "count"),
    avg_tfidf=("tfidf_similarity", "mean"),
    avg_semantic=("semantic_similarity", "mean"),
    avg_combined=("combined_score", "mean"),
).round(3).sort_values("avg_combined", ascending=False)

collection_relevance

## 8. Keyword Overlap Analysis

For each query, what fraction of the query's meaningful words appear in the retrieved source text? This is a simple, interpretable proxy for relevance.

In [ ]:
STOP_WORDS = {
    "a", "an", "the", "is", "was", "were", "are", "be", "been", "being",
    "have", "has", "had", "do", "does", "did", "will", "would", "could",
    "should", "may", "might", "shall", "can", "to", "of", "in", "for",
    "on", "with", "at", "by", "from", "as", "into", "about", "between",
    "through", "during", "before", "after", "and", "but", "or", "not",
    "no", "if", "then", "than", "that", "this", "these", "those",
    "it", "its", "he", "she", "his", "her", "him", "they", "them",
    "their", "we", "us", "our", "you", "your", "i", "me", "my",
    "what", "which", "who", "whom", "how", "when", "where", "why",
    "tell", "me", "did", "any", "some",
}

def keyword_overlap(question, source_text):
    """Fraction of query keywords found in the source text."""
    q_words = set(re.findall(r'\b\w+\b', question.lower())) - STOP_WORDS
    if not q_words:
        return 0.0
    s_text_lower = source_text.lower()
    found = sum(1 for w in q_words if w in s_text_lower)
    return found / len(q_words)

pairs_df["keyword_overlap"] = pairs_df.apply(
    lambda r: keyword_overlap(r["question"], r["source_text"]), axis=1
)

print("Keyword Overlap Distribution:")
print(pairs_df["keyword_overlap"].describe().round(3))
print()

# Queries with 0% keyword overlap — potential misses
zero_overlap = pairs_df[pairs_df["keyword_overlap"] == 0]
if len(zero_overlap) > 0:
    print(f"\nPairs with zero keyword overlap: {len(zero_overlap)}")
    for _, row in zero_overlap.head(5).iterrows():
        print(f"  Q: {row['question']}")
        print(f"  Source: [{row['collection']}] {row['title']}")
        print(f"  Semantic sim: {row['semantic_similarity']:.3f}")
        print()
else:
    print("\nNo pairs with zero keyword overlap.")

## 9. Qualitative Inspection: Weakest Retrievals

Side-by-side view of queries and their lowest-scoring source chunks. Read these to judge whether the retriever is finding genuinely useful content.

In [ ]:
# Show the N weakest query-source pairs
N_WEAKEST = 10

weakest = pairs_df.nsmallest(N_WEAKEST, "combined_score")

for i, (_, row) in enumerate(weakest.iterrows()):
    print("=" * 90)
    print(f"WEAK RETRIEVAL #{i+1}")
    print(f"  Combined: {row['combined_score']:.3f}  |  "
          f"TF-IDF: {row['tfidf_similarity']:.3f}  |  "
          f"Semantic: {row['semantic_similarity']:.3f}  |  "
          f"Keyword overlap: {row['keyword_overlap']:.0%}")
    print(f"  Intent: {row['intent']}")
    print("=" * 90)
    print(f"\nQUERY: {row['question']}")
    print(f"\nSOURCE [{row['collection']}]: {row['title']}")
    print("-" * 60)
    # Show first 500 chars of source text
    text = row['source_text']
    print(text[:500])
    if len(text) > 500:
        print(f"... ({len(text)} chars total)")
    print()

## 10. Qualitative Inspection: Strongest Retrievals

For comparison, the highest-scoring pairs — what does a good retrieval look like?

In [ ]:
N_STRONGEST = 5

strongest = pairs_df.nlargest(N_STRONGEST, "combined_score")

for i, (_, row) in enumerate(strongest.iterrows()):
    print("=" * 90)
    print(f"STRONG RETRIEVAL #{i+1}")
    print(f"  Combined: {row['combined_score']:.3f}  |  "
          f"TF-IDF: {row['tfidf_similarity']:.3f}  |  "
          f"Semantic: {row['semantic_similarity']:.3f}  |  "
          f"Keyword overlap: {row['keyword_overlap']:.0%}")
    print(f"  Intent: {row['intent']}")
    print("=" * 90)
    print(f"\nQUERY: {row['question']}")
    print(f"\nSOURCE [{row['collection']}]: {row['title']}")
    print("-" * 60)
    text = row['source_text']
    print(text[:500])
    if len(text) > 500:
        print(f"... ({len(text)} chars total)")
    print()

## 11. Full Interaction Deep Dive

Pick a specific query and see all its retrieved sources with scores, for a complete picture of what the RAG system found.

In [ ]:
# --- Set the query to inspect (use index from query_scores or set to None for the weakest) ---
QUERY_TEXT = None  # Set to a question string, or None for the weakest-scoring query

if QUERY_TEXT is None:
    QUERY_TEXT = query_scores.iloc[0]["question"]

interaction = pairs_df[pairs_df["question"] == QUERY_TEXT].sort_values(
    "combined_score", ascending=False
)

print(f"Query: \"{QUERY_TEXT}\"")
print(f"Intent: {interaction.iloc[0]['intent']}")
print(f"Sources retrieved: {len(interaction)}")
print(f"Avg combined score: {interaction['combined_score'].mean():.3f}")
print()

for i, (_, row) in enumerate(interaction.iterrows()):
    print(f"--- Source {i+1} (combined={row['combined_score']:.3f}, "
          f"semantic={row['semantic_similarity']:.3f}, "
          f"tfidf={row['tfidf_similarity']:.3f}, "
          f"keyword={row['keyword_overlap']:.0%}) ---")
    print(f"[{row['collection']}] {row['title']}")
    print()
    text = row['source_text']
    print(text[:600])
    if len(text) > 600:
        print(f"... ({len(text)} chars total)")
    print()

## 12. Similarity Scatter: TF-IDF vs Semantic

Pairs in the bottom-left are weak on both measures (likely irrelevant). Pairs where scores diverge reveal interesting cases — semantically related but lexically different, or keyword matches without topical relevance.

In [ ]:
# Text-based scatter overview
print("TF-IDF vs Semantic Similarity — Quadrant Breakdown")
print("=" * 50)

tfidf_median = pairs_df["tfidf_similarity"].median()
semantic_median = pairs_df["semantic_similarity"].median()

high_both = pairs_df[(pairs_df["tfidf_similarity"] >= tfidf_median) & (pairs_df["semantic_similarity"] >= semantic_median)]
high_sem_low_tfidf = pairs_df[(pairs_df["tfidf_similarity"] < tfidf_median) & (pairs_df["semantic_similarity"] >= semantic_median)]
low_sem_high_tfidf = pairs_df[(pairs_df["tfidf_similarity"] >= tfidf_median) & (pairs_df["semantic_similarity"] < semantic_median)]
low_both = pairs_df[(pairs_df["tfidf_similarity"] < tfidf_median) & (pairs_df["semantic_similarity"] < semantic_median)]

print(f"\nMedians — TF-IDF: {tfidf_median:.3f}, Semantic: {semantic_median:.3f}")
print(f"\n  High TF-IDF + High Semantic (strong match):        {len(high_both):>4} ({len(high_both)/len(pairs_df)*100:.0f}%)")
print(f"  Low TF-IDF + High Semantic (meaning match only):    {len(high_sem_low_tfidf):>4} ({len(high_sem_low_tfidf)/len(pairs_df)*100:.0f}%)")
print(f"  High TF-IDF + Low Semantic (keyword match only):    {len(low_sem_high_tfidf):>4} ({len(low_sem_high_tfidf)/len(pairs_df)*100:.0f}%)")
print(f"  Low TF-IDF + Low Semantic (weak match):             {len(low_both):>4} ({len(low_both)/len(pairs_df)*100:.0f}%)")

In [ ]:
# Interesting cases: high semantic but low TF-IDF (meaning match without keyword overlap)
print("\nHigh semantic / Low TF-IDF — meaning match without shared vocabulary:")
print("(These show the embedding model finding relevant content that keywords would miss)\n")

interesting = high_sem_low_tfidf.nlargest(5, "semantic_similarity")
for _, row in interesting.iterrows():
    print(f"  Q: {row['question']}")
    print(f"  Source: [{row['collection']}] {row['title']}")
    print(f"  Semantic: {row['semantic_similarity']:.3f}  TF-IDF: {row['tfidf_similarity']:.3f}")
    print(f"  Text preview: {row['source_text'][:150]}...")
    print()

## 13. Source Text Length vs Relevance

Are shorter or longer chunks more relevant? This can inform chunking strategy.

In [ ]:
pairs_df["source_length"] = pairs_df["source_text"].str.len()
pairs_df["source_word_count"] = pairs_df["source_text"].str.split().str.len()

# Bin by length
bins = [0, 200, 500, 1000, 2000, float("inf")]
labels = ["<200", "200-500", "500-1000", "1000-2000", "2000+"]
pairs_df["length_bucket"] = pd.cut(pairs_df["source_length"], bins=bins, labels=labels)

length_relevance = pairs_df.groupby("length_bucket", observed=True).agg(
    count=("question", "count"),
    avg_semantic=("semantic_similarity", "mean"),
    avg_tfidf=("tfidf_similarity", "mean"),
    avg_combined=("combined_score", "mean"),
    avg_keyword_overlap=("keyword_overlap", "mean"),
).round(3)

length_relevance

## 14. Summary Dashboard

In [ ]:
print("RETRIEVAL RELEVANCE SUMMARY")
print("=" * 50)
print(f"Total query-source pairs analyzed: {len(pairs_df)}")
print(f"Unique queries: {pairs_df['question'].nunique()}")
print(f"Unique source documents: {pairs_df['parent_id'].nunique()}")
print()
print(f"Avg semantic similarity:  {pairs_df['semantic_similarity'].mean():.3f}")
print(f"Avg TF-IDF similarity:    {pairs_df['tfidf_similarity'].mean():.3f}")
print(f"Avg keyword overlap:      {pairs_df['keyword_overlap'].mean():.1%}")
print(f"Avg combined score:       {pairs_df['combined_score'].mean():.3f}")
print()

# Flag potentially problematic queries (combined < 0.1)
weak_threshold = 0.1
weak_queries = query_scores[query_scores["avg_combined"] < weak_threshold]
print(f"Queries with avg combined score < {weak_threshold}: {len(weak_queries)}")
if len(weak_queries) > 0:
    for _, row in weak_queries.iterrows():
        print(f"  [{row['intent']}] \"{row['question']}\" (score: {row['avg_combined']:.3f})")